In [1]:
# %% [markdown]
# # Weekly Bond Direction Model
# 
# This notebook builds a logistic regression model to predict **next week's Open→Close direction**
# in bond futures (or any other instrument with OHLC data).
# 
# It:
# - Converts daily prices to weekly OHLC candles
# - Engineers momentum, volatility, and candle-structure features
# - Runs a walk-forward backtest
# - Reports hit rate, Sharpe, and AUC
# - Outputs a live signal for the next week
# 
# Just set `csv_path` below to your local OHLC file.

# %%
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
from pathlib import Path

# %%
# === USER INPUT ===
csv_path = r"C:\Users\loci_\Desktop\trading_webapp\DATA\all_input_files\TY1.csv"  # <-- change this
min_train_weeks = 156  # 3 years of weekly data for training

# %%
def load_prices(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    cols = {c.lower(): c for c in df.columns}
    for r in ['date', 'open', 'high', 'low', 'close']:
        if r not in cols:
            raise ValueError(f"Missing required column: {r}. Found {list(df.columns)}")
    df = df.rename(columns={
        cols['date']: 'Date', cols['open']: 'Open', 
        cols['high']: 'High', cols['low']: 'Low', cols['close']: 'Close'
    })
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    return df

def to_weekly(df: pd.DataFrame) -> pd.DataFrame:
    df = df.set_index('Date')
    weekly = pd.DataFrame({
        'Open': df['Open'].resample('W-FRI').first(),
        'High': df['High'].resample('W-FRI').max(),
        'Low':  df['Low'].resample('W-FRI').min(),
        'Close': df['Close'].resample('W-FRI').last(),
    })
    weekly = weekly.dropna(subset=['Open', 'High', 'Low', 'Close'])
    weekly = weekly.reset_index().rename(columns={'Date': 'WeekEnd'})
    return weekly

def engineer_features(w: pd.DataFrame):
    df = w.copy()
    df['body'] = (df['Close'] - df['Open'])
    df['range'] = (df['High'] - df['Low']).replace(0, np.nan)
    df['body_pct_range'] = df['body'] / df['range']
    df['ret_oc'] = df['Close'] / df['Open'] - 1
    df['gap_o_prev_c'] = df['Open'].pct_change()
    df['ret_cc'] = df['Close'].pct_change()
    df['mom_4w'] = df['Close'].pct_change(4)
    df['mom_8w'] = df['Close'].pct_change(8)
    df['vol_4w'] = df['ret_cc'].rolling(4).std()
    df['vol_8w'] = df['ret_cc'].rolling(8).std()
    df['body_lag1'] = df['body'].shift(1)
    df['body_lag2'] = df['body'].shift(2)
    df['sign_body_lag1'] = np.sign(df['body_lag1'])
    weeknum = df['WeekEnd'].dt.isocalendar().week.astype(int)
    df['wk_sin'] = np.sin(2*np.pi*weeknum/52)
    df['wk_cos'] = np.cos(2*np.pi*weeknum/52)
    df['next_open'] = df['Open'].shift(-1)
    df['next_close'] = df['Close'].shift(-1)
    df['y_ret'] = df['next_close'] / df['next_open'] - 1
    df['y'] = (df['y_ret'] > 0).astype(int)

    feature_cols = [
        'body', 'body_pct_range', 'ret_oc', 'gap_o_prev_c', 'ret_cc',
        'mom_4w', 'mom_8w', 'vol_4w', 'vol_8w',
        'body_lag1', 'body_lag2', 'sign_body_lag1', 'wk_sin', 'wk_cos'
    ]
    df = df.dropna(subset=feature_cols + ['y']).reset_index(drop=True)
    return df, feature_cols

# %%
def walk_forward_backtest(df: pd.DataFrame, feature_cols: list, min_train_weeks: int = 156) -> pd.DataFrame:
    X = df[feature_cols].values
    y = df['y'].values
    dates = df['WeekEnd'].values
    oc_ret_next = df['y_ret'].values

    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000))
    ])
    rows = []
    for t in range(min_train_weeks, len(df)-1):
        X_train, y_train = X[:t], y[:t]
        X_pred = X[t:t+1]
        d_pred = dates[t]
        pipe.fit(X_train, y_train)
        proba = pipe.predict_proba(X_pred)[0, 1]
        pred = int(proba >= 0.5)
        realized = oc_ret_next[t]
        rows.append({
            'WeekEnd': pd.to_datetime(d_pred),
            'proba_up': float(proba),
            'pred_up': pred,
            'realized_ret_oc_next': float(realized),
            'hit': int((pred==1 and realized>0) or (pred==0 and realized<=0)),
            'strategy_ret': float((1 if pred==1 else -1) * realized)
        })
    return pd.DataFrame(rows)

# %%
def summarize(bt: pd.DataFrame):
    acc = bt['hit'].mean()
    avg_ret = bt['strategy_ret'].mean()
    vol = bt['strategy_ret'].std()
    sharpe = (avg_ret / vol) * np.sqrt(52) if vol > 0 else np.nan
    cum = (1 + bt['strategy_ret']).prod() - 1
    print(f"Hit rate: {acc:.2%}")
    print(f"Avg weekly return: {avg_ret:.4%}")
    print(f"Volatility (weekly): {vol:.4%}")
    print(f"Sharpe (annualized): {sharpe:.2f}")
    print(f"Cumulative strategy return: {cum:.2%}")
    print()
    y_true = (bt['realized_ret_oc_next'] > 0).astype(int)
    auc = roc_auc_score(y_true, bt['proba_up']) if len(np.unique(y_true)) > 1 else np.nan
    print(f"AUC: {auc:.3f}")
    print(classification_report(y_true, bt['pred_up'], digits=3))

# %%
def predict_next_week(df: pd.DataFrame, feature_cols: list):
    X = df[feature_cols].values
    y = df['y'].values
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
    pipe.fit(X[:-1], y[:-1])
    proba = pipe.predict_proba(X[-1:])[0, 1]
    pred = int(proba >= 0.5)
    last_week = df['WeekEnd'].iloc[-1]
    print(f"Signal generated on week ending {last_week.date()}:")
    print(f"Prob Up: {proba:.2%} → Prediction: {'UP' if pred==1 else 'DOWN'} (for NEXT week’s Open→Close)")

# %%
raw = load_prices(csv_path)
weekly = to_weekly(raw)
feats, feature_cols = engineer_features(weekly)

bt = walk_forward_backtest(feats, feature_cols, min_train_weeks)
summarize(bt)
predict_next_week(feats, feature_cols)

# %%
plt.figure(figsize=(10,5))
bt['cum_strategy'] = (1+bt['strategy_ret']).cumprod() - 1
plt.plot(bt['WeekEnd'], bt['cum_strategy'])
plt.title("Cumulative Strategy Return (Weekly Bond Direction Model)")
plt.xlabel("Date")
plt.ylabel("Cumulative Return")
plt.grid(True)
plt.show()


ValueError: Missing required column: open. Found ['Date', 'PX_OPEN', 'PX_HIGH', 'PX_LOW', 'PX_CLOSE_1D', 'PX_VOLUME', 'OPEN_INT', 'INSTRUMENT', 'SECTYPE', 'EXCHANGE', 'CRNCY', 'TICK_SIZE', 'TICK_VALUE', 'POINT_VALUE', 'CONTRACT_VALUE', 'near', 'far', 'st_dev', 'no_days', 'no_days.1', 'Standard Cost', 'Instrument_Weights', 'Exchange rate']